# refactor

> Exact text edits over a sandbox: preview a plan, then apply the file contents it approved.

In [ ]:
#| default_exp refactor

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import tempfile
from fastcore.test import test_eq, test_fail

In [ ]:
#| export
from __future__ import annotations
import ast, builtins, re, textwrap
from dataclasses import dataclass
from functools import partial
from itertools import accumulate
from fastcore.basics import first
from fastcore.xtras import Path

## What an edit plan is

`Sandbox` in `core` decides which paths an agent may touch, and `apply_edits` applies exact-text
edits to one file. This module sits between them: it reads a workspace and returns the whole
`before` and `after` of every file a change would touch, so a person sees the change before it
happens. Nothing here writes.

`is_texty` answers a different question from `SKIP_SUFFIXES` in `host`. That one is what an index
skips; this one is what a text editor can open at all, so `.ico`, `.ttf` and `.mp4` are in it and
`.o`, `.bin` and `.gguf` are not.

In [ ]:
#| export
#: Suffixes no text editor opens. Not `SKIP_SUFFIXES`: that is what an index walks past.
_BINARY = {'.png', '.jpg', '.jpeg', '.gif', '.webp', '.ico', '.pdf', '.zip', '.gz', '.tar', '.whl', '.so', '.dylib',
    '.dll', '.pyc', '.parquet', '.db', '.sqlite', '.woff', '.woff2', '.ttf', '.mp4', '.mp3', '.npy', '.npz', '.pkl'}

def is_texty(p):
    "Whether this can be opened in a text editor at all."
    return Path(p).suffix.lower() not in _BINARY

In [ ]:
test_eq(is_texty('a.py'), True)
test_eq(is_texty('a.PNG'), False)   # suffix match is case-insensitive
test_eq(is_texty('README'), True)

In [ ]:
#| export
@dataclass
class FileEdit:
    path: str
    before: str
    after: str
    edits: int
    def dict(self):
        return dict(path=self.path, before=self.before, after=self.after, edits=self.edits)

In [ ]:
e = FileEdit('a.py', 'x = 1\n', 'y = 1\n', 1)
test_eq(e.dict()['path'], 'a.py')
test_eq(e.dict()['edits'], 1)

## Replace across a workspace

`replace_plan` takes anything with `walk()` and `read(path)`, which is the `Host` interface, and
caps the walk itself rather than asking a host to. A `Host.read` answers None for a file it cannot
read, so that joins the two other reasons to skip one: a notebook, and text that came back with a
replacement character standing in for bytes a rewrite would lose. Notebooks and binaries are skipped rather than mangled, and so is any file that reads
back with a replacement character, because rewriting one would lose the bytes it stood for.

In [ ]:
#| export
def _pattern(query, regex=False, case=False, word=False):
    if not query: raise ValueError('find text is empty')
    text = query if regex else re.escape(query)
    if word: text = r'\b' + text + r'\b'
    try: return re.compile(text, 0 if case else re.I)
    except re.error as e: raise ValueError(f'invalid regular expression: {e}') from e

def replace_plan(fs, query, replacement, regex=False, case=False, word=False, limit=20000):
    "Changed editable workspace files for one literal or regular-expression replacement."
    pat = _pattern(str(query), bool(regex), bool(case), bool(word))
    repl = str(replacement) if regex else str(replacement).replace('\\', '\\\\')
    rows, skipped = [], []
    for path in fs.walk()[:limit]:
        if path.suffix.lower() == '.ipynb' or not is_texty(path):
            skipped.append(str(path)); continue
        try: before = fs.read(path)
        except Exception: skipped.append(str(path)); continue
        if before is None or '\ufffd' in before: skipped.append(str(path)); continue
        try: after, n = pat.subn(repl, before)
        except re.error as e: raise ValueError(f'invalid replacement: {e}') from e
        if n: rows.append(FileEdit(str(path), before, after, n))
    return rows, skipped

In [ ]:
class _Fs:
    "The two methods `replace_plan` needs, which is what `Host` already gives it."
    def __init__(self, d): self.d = Path(d)
    def walk(self): return sorted(p for p in self.d.rglob('*') if p.is_file())
    def read(self, p): return Path(p).read_text()

d = Path(tempfile.mkdtemp())
(d/'a.py').write_text('cat = 1\nCat = 2\nconcat = 3\n')
(d/'b.png').write_bytes(b'\x89PNG')
fs = _Fs(d)

In [ ]:
rows, skipped = replace_plan(fs, 'cat', 'dog')
test_eq([r.path for r in rows], [str(d/'a.py')])
test_eq(rows[0].after, 'dog = 1\ndog = 2\ncondog = 3\n')   # case-insensitive and unanchored by default
test_eq(rows[0].edits, 3)
test_eq(skipped, [str(d/'b.png')])                        # binaries are skipped, never rewritten

In [ ]:
test_eq(replace_plan(fs, 'cat', 'dog', case=True)[0][0].edits, 2)
test_eq(replace_plan(fs, 'cat', 'dog', word=True)[0][0].edits, 2)
test_eq(replace_plan(fs, r'c.t', 'dog', regex=True)[0][0].edits, 3)
test_eq(replace_plan(fs, 'nothing here', 'x')[0], [])
test_fail(lambda: replace_plan(fs, '', 'x'), contains='find text is empty')
test_fail(lambda: replace_plan(fs, '[', 'x', regex=True), contains='invalid regular expression')

In [ ]:
#: A literal replacement is taken literally, so a backslash in it survives.
test_eq(replace_plan(fs, 'cat', r'a\b')[0][0].after.splitlines()[0], r'a\b = 1')

In [ ]:
#: A `Host.read` answers None for a file it cannot read. That is a skip, not a crash.
class _Unreadable(_Fs):
    def read(self, p): return None
test_eq(replace_plan(_Unreadable(d), 'cat', 'dog'), ([], [str(d/'a.py'), str(d/'b.png')]))

## Extract and inline

`extract` lifts a selection into a variable, a constant, a function or a method. Every case it
cannot do safely raises rather than guessing: a selection that returns or yields, a constant off
module scope, a partial statement line. `inline` goes the other way, and needs the variable
assigned exactly once on lines of its own.

In [ ]:
#| export
def _name(name):
    if not str(name).isidentifier(): raise ValueError('name must be a valid Python identifier')
    return str(name)

def _line(source, pos): return source.count('\n', 0, pos), source.rfind('\n', 0, pos) + 1

def _indent(source, pos):
    _, start = _line(source, pos)
    return re.match(r'[ \t]*', source[start:]).group()

def _expr(text):
    try: return ast.parse(text, mode='eval').body
    except SyntaxError as e: raise ValueError('select one complete Python expression') from e

In [ ]:
#| export
def extract(path, source, start, end, kind, name):
    "One safe single-file extraction; unsupported selections fail rather than guess."
    start, end, name = int(start), int(end), _name(name)
    if not 0 <= start < end <= len(source): raise ValueError('select code before extracting')
    text, indent = source[start:end], _indent(source, start)
    if kind in {'variable', 'constant'}:
        node = _expr(text)
        if kind == 'constant':
            try: ast.literal_eval(node)
            except Exception as e: raise ValueError('constants must be literal values') from e
            name = name.upper()
            if not re.fullmatch(r'[A-Z_][A-Z0-9_]*', name): raise ValueError('constant names use UPPER_CASE')
            if indent: raise ValueError('extract constants from module scope')
            tree = ast.parse(source); at = 0
            if tree.body and isinstance(tree.body[0], ast.Expr) and isinstance(tree.body[0].value, ast.Constant): at = tree.body[0].end_lineno
            while at < len(source.splitlines()) and source.splitlines()[at].startswith(('import ', 'from ')): at += 1
            point = sum(len(x) + 1 for x in source.splitlines()[:at])
            after = source[:point] + f'{name} = {text}\n' + source[point:start] + name + source[end:]
        else:
            line, line_start = _line(source, start)
            after = source[:line_start] + f'{indent}{name} = {text}\n' + source[line_start:start] + name + source[end:]
        return FileEdit(path, source, after, 1)
    if kind not in {'function', 'method'}: raise ValueError('unknown extraction')
    if source[_line(source, start)[1]:start].strip() or source[end:source.find('\n', end) if source.find('\n', end) >= 0 else len(source)].strip():
        raise ValueError('functions require complete statement lines')
    # the selection carries its own indentation, and `ast` will not parse an indented block
    block = textwrap.dedent(text)
    try: selected = ast.parse(block).body
    except SyntaxError as e: raise ValueError(f'select whole statements to extract ({e.msg})') from e
    if not selected: raise ValueError('select statements')
    if any(isinstance(n, (ast.Return, ast.Yield, ast.YieldFrom)) for node in selected for n in ast.walk(node)):
        raise ValueError('a selection that returns or yields cannot be lifted into its own function')
    params, stores = _reads_before_writes(selected), _writes(selected)
    params = [p for p in params if p not in set(dir(builtins))]
    returns = sorted(n for n in stores if n in _reads(source[end:]))
    body = ''.join(indent + '    ' + line if line.strip() else line for line in block.splitlines(True))
    if not body.endswith('\n'): body += '\n'   # a selection can end mid-line; a body cannot
    if returns: body += f'{indent}    return {", ".join(returns)}\n'
    args = list(params)
    if kind != 'method':
        call, head = f'{name}({", ".join(args)})', ', '.join(args)
        if returns: call = f'{", ".join(returns)} = {call}'
        definition = f'{indent}def {name}({head}):\n{body}'
        return FileEdit(path, source, source[:start] + definition + '\n' + indent + call + '\n' + source[end:], 1)
    # a method has to be a sibling of the one it came out of: nested inside it, `self.name` is
    # not an attribute of anything
    args = [p for p in args if p != 'self']
    call = f'self.{name}({", ".join(args)})'
    if returns: call = f'{", ".join(returns)} = {call}'
    holder, member = _enclosing_method(source, start)
    body = textwrap.indent(textwrap.dedent(body), member + '    ')
    definition = f'\n\n{member}def {name}(self{", " if args else ""}{", ".join(args)}):\n{body.rstrip()}\n'
    after = source[:start] + indent + call + '\n' + source[end:]
    at = _offset_after(after, holder)
    return FileEdit(path, source, after[:at] + definition + after[at:], 1)

In [ ]:
#| export
def _enclosing_method(source, pos):
    "The method `pos` sits in and the indent its siblings use, or a reason there is none."
    tree, starts = ast.parse(source), _starts(source)
    for cls in [n for n in ast.walk(tree) if isinstance(n, ast.ClassDef)]:
        for fn in [n for n in cls.body if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))]:
            begin, stop = _at(source, starts, fn)
            if begin <= pos <= stop: return fn, ' ' * fn.col_offset
    raise ValueError('extract a method from inside one of a class\'s own methods')

def _offset_after(source, node):
    "Where a sibling of `node` goes: the end of the statement it follows."
    starts = _starts(source)
    for fn in [n for n in ast.walk(ast.parse(source))
               if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef)) and n.name == node.name]:
        return _at(source, starts, fn)[1]
    return len(source)

def _writes(nodes):
    "Every name the selection binds."
    return {n.id for node in nodes for n in ast.walk(node)
            if isinstance(n, ast.Name) and isinstance(n.ctx, (ast.Store, ast.AugStore))} | {
            n.id for node in nodes for n in ast.walk(node)
            if isinstance(n, ast.Name) and isinstance(n.ctx, ast.Store)}

def _reads(text):
    "Every name the text loads, so a name still wanted afterwards can be told from a dead one."
    for candidate in (text, textwrap.dedent(text)):
        try: tree = ast.parse(candidate)
        except SyntaxError: continue
        return {n.id for n in ast.walk(tree) if isinstance(n, ast.Name) and isinstance(n.ctx, ast.Load)}
    return set(re.findall(r'\b[A-Za-z_][A-Za-z0-9_]*\b', text))

#: Where a statement reads before it binds. `ast.walk` is breadth-first, so it would see the target
#: of `total = total + 1` before the value and call the name already bound.
_EVAL_ORDER = {ast.Assign: ('value', 'targets'), ast.AugAssign: ('value', 'target'),
               ast.AnnAssign: ('value', 'target'), ast.For: ('iter', 'target'),
               ast.AsyncFor: ('iter', 'target'), ast.comprehension: ('iter', 'target'),
               ast.withitem: ('context_expr', 'optional_vars')}

In [ ]:
#| export
def _reads_before_writes(nodes):
    "Names the selection reads before it binds them: those have to arrive as parameters."
    bound, params = set(), []
    def visit(node):
        if isinstance(node, ast.Name):
            if isinstance(node.ctx, ast.Load):
                if node.id not in bound and node.id not in params: params.append(node.id)
            else: bound.add(node.id)
            return
        if (order := _EVAL_ORDER.get(type(node))) is not None:
            for field in order:
                for child in _children(getattr(node, field, None)): visit(child)
            for name, value in ast.iter_fields(node):
                if name in order: continue
                for child in _children(value): visit(child)
            return
        for child in ast.iter_child_nodes(node): visit(child)
    for node in nodes: visit(node)
    return sorted(params)

def _children(value):
    if isinstance(value, list): return [x for x in value if isinstance(x, ast.AST)]
    return [value] if isinstance(value, ast.AST) else []

def _starts(source):
    "Character offset at which each line begins, plus the end of the text."
    return list(accumulate(map(len, source.splitlines(True)), initial=0))

def _at(source, starts, node):
    "A node's (start, end) character offsets. `col_offset` counts utf-8 bytes, not characters."
    def one(lineno, col):
        begin = starts[lineno - 1]
        return begin + len(source[begin:starts[lineno]].encode()[:col].decode('utf-8', 'ignore'))
    return one(node.lineno, node.col_offset), one(node.end_lineno, node.end_col_offset)

_ATOMIC = (ast.Name, ast.Constant, ast.Attribute, ast.Subscript, ast.Call,
           ast.List, ast.Dict, ast.Set, ast.ListComp, ast.DictComp, ast.SetComp, ast.JoinedStr)

In [ ]:
#| export
def _binding(tree, name):
    "The one plain `name = value` statement that defines `name`, or a reason there isn't one."
    if any(name in n.names for n in ast.walk(tree) if isinstance(n, (ast.Global, ast.Nonlocal))):
        raise ValueError(f'{name} is declared global or nonlocal')
    stores = [n for n in ast.walk(tree) if isinstance(n, ast.Name) and n.id == name
              and isinstance(n.ctx, (ast.Store, ast.Del))]
    plain = [n for n in ast.walk(tree) if isinstance(n, ast.Assign) and len(n.targets) == 1
             and isinstance(n.targets[0], ast.Name) and n.targets[0].id == name]
    if len(plain) != 1 or len(stores) != 1:
        raise ValueError(f'inline needs {name} assigned exactly once, as a plain `{name} = value`')
    return plain[0]

def inline(path, source, pos):
    "Replace a variable's uses with its value and delete the assignment."
    pos = int(pos)
    if not 0 <= pos <= len(source): raise ValueError('put the cursor on a variable first')
    tree, starts = ast.parse(source), _starts(source)
    at = partial(_at, source, starts)
    spans = {n: at(n) for n in ast.walk(tree) if isinstance(n, ast.Name)}
    here = first(n for n, (a, b) in spans.items() if a <= pos <= b)
    if here is None: raise ValueError('put the cursor on a variable first')
    assign = _binding(tree, here.id)
    astart, aend = at(assign)
    lstart, lend = starts[assign.lineno - 1], starts[assign.end_lineno]
    if source[lstart:astart].strip() or source[aend:lend].strip():
        raise ValueError('inline needs the assignment on lines of its own')
    value = source[slice(*at(assign.value))]
    if not isinstance(assign.value, _ATOMIC): value = f'({value})'
    uses = [span for n, span in spans.items() if n.id == here.id and isinstance(n.ctx, ast.Load)]
    if not uses: raise ValueError(f'{here.id} is never read, so there is nothing to inline')
    if any(a < aend for a, _ in uses): raise ValueError(f'{here.id} is read before it is assigned')
    after = source
    for a, b in sorted(uses, reverse=True): after = after[:a] + value + after[b:]
    return FileEdit(path, source, after[:lstart] + after[lend:], len(uses))

In [ ]:
s = 'def f():\n    total = 2 + 3\n    return total\n'
test_eq(extract('a.py', s, s.index('2 + 3'), s.index('2 + 3')+5, 'variable', 'n').after,
        'def f():\n    n = 2 + 3\n    total = n\n    return total\n')

In [ ]:
c = 'x = 42\n'
test_eq(extract('a.py', c, 4, 6, 'constant', 'n').after, 'N = 42\nx = N\n')
test_eq(extract('a.py', c, 4, 6, 'constant', 'lower').after, 'LOWER = 42\nx = LOWER\n')
test_fail(lambda: extract('a.py', 'x = f()\n', 4, 7, 'constant', 'N'), contains='literal values')
inner = 'def f():\n    x = 42\n'
test_fail(lambda: extract('a.py', inner, 17, 19, 'constant', 'N'), contains='module scope')

In [ ]:
b = 'a = 1\nb = a + 1\nprint(b)\n'
out = extract('a.py', b, 0, b.index('print')-1, 'function', 'setup').after
test_eq(out, 'def setup():\n    a = 1\n    b = a + 1\n    return b\n\nb = setup()\n\nprint(b)\n')

In [ ]:
#: A selection that returns cannot become a function of its own.
r = 'def f():\n    return 1\n'
test_fail(lambda: extract('a.py', r, r.index('    return'), len(r), 'function', 'g'),
          contains='returns or yields')
test_fail(lambda: extract('a.py', b, 1, 4, 'function', 'g'), contains='complete statement lines')
test_fail(lambda: extract('a.py', b, 0, 5, 'unknown', 'g'), contains='unknown extraction')
test_fail(lambda: extract('a.py', b, 0, 5, 'variable', '2bad'), contains='valid Python identifier')

In [ ]:
m = 'class C:\n    def f(self):\n        n = 1 + 1\n        print(n)\n'
out = extract('a.py', m, m.index('        n = 1'), m.index('        print')-1, 'method', 'setup').after
test_eq('    def setup(self):' in out, True)         # a sibling of the method it came from, not a nested def
test_eq('n = self.setup()' in out, True)
test_fail(lambda: extract('a.py', b, 0, 5, 'method', 'g'), contains="class's own methods")

In [ ]:
i = 'n = 1 + 1\nprint(n)\n'
test_eq(inline('a.py', i, 0).after, 'print((1 + 1))\n')   # a non-atomic value keeps its brackets
test_eq(inline('a.py', 'n = 1\nprint(n)\n', 0).after, 'print(1)\n')
test_eq(inline('a.py', i, 0).edits, 1)

In [ ]:
test_fail(lambda: inline('a.py', 'n = 1\nn = 2\nprint(n)\n', 0), contains='exactly once')
test_fail(lambda: inline('a.py', 'n = 1\n', 0), contains='never read')
test_fail(lambda: inline('a.py', '1 + 1\n', 0), contains='put the cursor on a variable')
g = 'def f():\n    global n\n    n = 1\nprint(n)\n'
test_fail(lambda: inline('a.py', g, g.index('print(n)')+6), contains='global or nonlocal')

## Move a definition, and repoint what imported it

`move_plan` takes top-level functions and classes out of one file, puts them in another, and
rewrites every `from src import name` and `src.name` in the files it is given. It refuses rather
than producing a broken tree: a name the destination already binds, a move that would make the two
modules import each other, and any file that would end up reading a name nothing gives it.

In [ ]:
#| export
_BUILTINS = frozenset(dir(builtins))

def module_of(path):
    "Dotted module name for a file, from the top of its `__init__.py` chain."
    p = Path(path)
    parts, d = ([] if p.stem == '__init__' else [p.stem]), p.parent
    while (d/'__init__.py').is_file(): parts.append(d.name); d = d.parent
    return '.'.join(reversed(parts))

def _parse(path, source):
    try: return ast.parse(source)
    except SyntaxError as e: raise ValueError(f'{Path(path).name} does not parse: {e.msg} (line {e.lineno})') from e

def _absolute(node, here):
    "The absolute module an `ImportFrom` names, resolving `level` against the module holding it."
    if not node.level: return node.module or ''
    return '.'.join([*here.split('.')[:-node.level], *([node.module] if node.module else [])])

def _stmt(target, here, names):
    "A `from ... import` reaching `target` from `here`, relative where a shared package allows one."
    tp, hp = target.split('.'), here.split('.')[:-1]
    n = 0
    while n < len(tp) - 1 and n < len(hp) and tp[n] == hp[n]: n += 1
    dots = '.' * (len(hp) - n + 1) if n else ''
    return f"from {dots}{'.'.join(tp[n:])} import {', '.join(names)}"

def _spec(alias): return f'{alias.name} as {alias.asname}' if alias.asname else alias.name
def _from(node, aliases): return f"from {'.' * node.level}{node.module or ''} import {', '.join(_spec(a) for a in aliases)}"

In [ ]:
#| export
def _bound(tree):
    "Every name a module binds at its top level, mapped to the statement that binds it."
    out = {}
    for n in tree.body:
        if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)): out[n.name] = n
        elif isinstance(n, (ast.Import, ast.ImportFrom)):
            for a in n.names: out[(a.asname or a.name).split('.')[0]] = n
        elif isinstance(n, ast.Assign):
            for t in n.targets:
                for x in ast.walk(t):
                    if isinstance(x, ast.Name): out[x.id] = n
        elif isinstance(n, ast.AnnAssign) and isinstance(n.target, ast.Name): out[n.target.id] = n
    return out

def _free(nodes):
    "Names the nodes read without binding, which is what has to reach them wherever they go."
    load, store = set(), set()
    for node in nodes:
        for n in ast.walk(node):
            if isinstance(n, ast.Name): (load if isinstance(n.ctx, ast.Load) else store).add(n.id)
            elif isinstance(n, ast.arg): store.add(n.arg)
            elif isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)): store.add(n.name)
            elif isinstance(n, ast.alias): store.add((n.asname or n.name).split('.')[0])
            elif isinstance(n, ast.ExceptHandler) and n.name: store.add(n.name)
            elif isinstance(n, ast.Global): store.update(n.names)
    return load - store - _BUILTINS

def _span(source, starts, node):
    "What a definition owns: the comments above it, its decorators, and the blank lines below."
    lines = source.splitlines()
    top = min([d.lineno for d in getattr(node, 'decorator_list', [])] + [node.lineno])
    while top > 1 and lines[top - 2].lstrip().startswith('#'): top -= 1
    end = node.end_lineno
    while end < len(lines) and not lines[end].strip(): end += 1
    return starts[top - 1], starts[end]

In [ ]:
#| export
def top_symbols(source):
    "Top-level functions and classes, each with the character span it owns."
    tree, starts = ast.parse(source), _starts(source)
    kind = lambda n: 'class' if isinstance(n, ast.ClassDef) else 'function'
    return [dict(name=n.name, kind=kind(n), line=n.lineno, **dict(zip(('start', 'end'), _span(source, starts, n))))
            for n in tree.body if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef))]

In [ ]:
src = 'import math\n\ndef area(r): return math.pi * r * r\n\nclass Box:\n    pass\n'
test_eq([(s['name'], s['kind']) for s in top_symbols(src)], [('area', 'function'), ('Box', 'class')])
test_eq(module_of('/tmp/nope/x.py'), 'x')

In [ ]:
#| export
def _apply(text, edits):
    for a, b, new in sorted(edits, reverse=True): text = text[:a] + new + text[b:]
    return text

def _import_point(tree, starts):
    "Where a new import goes: after the module docstring and the imports already following it."
    body, at, i = tree.body, 0, 0
    if body and isinstance(body[0], ast.Expr) and isinstance(getattr(body[0].value, 'value', None), str):
        at, i = starts[body[0].end_lineno], 1
    while i < len(body) and isinstance(body[i], (ast.Import, ast.ImportFrom)):
        at, i = starts[body[i].end_lineno], i + 1
    return at

def _all_edit(text, starts, tree, add=(), drop=()):
    "An edit rewriting a module's `__all__`, or None when it has none this can read."
    node = first(n for n in tree.body if isinstance(n, ast.Assign) and len(n.targets) == 1
                 and isinstance(n.targets[0], ast.Name) and n.targets[0].id == '__all__')
    if node is None or not isinstance(node.value, (ast.List, ast.Tuple)): return None
    cur = [e.value for e in node.value.elts if isinstance(e, ast.Constant) and isinstance(e.value, str)]
    if len(cur) != len(node.value.elts): return None
    new = [x for x in cur if x not in set(drop)] + [x for x in add if x not in cur]
    if new == cur: return None
    return (*_at(text, starts, node.value), '[' + ', '.join(repr(x) for x in new) + ']')

def _has_import(tree, here, module, spec):
    "The `ImportFrom` already naming `spec` from `module`, and every spec it carries."
    for n in ast.walk(tree):
        if isinstance(n, ast.ImportFrom) and _absolute(n, here) == module:
            have = {_spec(a) for a in n.names}
            if spec in have: return n, have
    return None, set()

def _add_imports(text, tree, starts, here, froms, plains):
    "Edits folding `froms` into whatever import block the file already has, plus the plain imports."
    edits, add = [], []
    plain_texts = {f"import {_spec(a)}" for n in ast.walk(tree) if isinstance(n, ast.Import) for a in n.names}
    for module, specs in sorted(froms.items()):
        if module == here: continue
        node, have = None, set()
        for spec in sorted(specs):
            node, have = _has_import(tree, here, module, spec)
            if node is not None: break
        else:
            node, have = first((n, {_spec(a) for a in n.names}) for n in ast.walk(tree)
                               if isinstance(n, ast.ImportFrom) and _absolute(n, here) == module) or (None, set())
        want = sorted(have | set(specs))
        if want == sorted(have): continue
        if node is not None: edits.append((*_at(text, starts, node), _stmt(module, here, want)))
        else: add.append(_stmt(module, here, sorted(specs)))
    add += [p for p in sorted(plains) if p not in plain_texts]
    if add:
        at = _import_point(tree, starts)
        edits.append((at, at, ''.join(s + '\n' for s in add)))
    return edits

def _carry(froms, plains, node, name, here):
    "Record the import that gave `name` to the source file, so the destination gains it too."
    if isinstance(node, ast.ImportFrom):
        alias = first(a for a in node.names if (a.asname or a.name).split('.')[0] == name)
        froms.setdefault(_absolute(node, here), set()).add(_spec(alias))
    else:
        alias = first(a for a in node.names if (a.asname or a.name).split('.')[0] == name)
        plains.add(f'import {_spec(alias)}')

In [ ]:
#| export
def _repoint(path, text, src_mod, dest_mod, names):
    "One file's imports of `names`, pointed at `dest_mod`."
    here, moved = module_of(path), set(names)
    tree, starts = _parse(path, text), _starts(text)
    bound, edits, hit, notes = _bound(tree), [], False, []
    for node in ast.walk(tree):
        if not isinstance(node, ast.ImportFrom) or _absolute(node, here) != src_mod: continue
        taken = [a for a in node.names if a.name in moved]
        if not taken: continue
        hit = True
        keep = [a for a in node.names if a.name not in moved]
        a, b = _at(text, starts, node)
        indent = text[starts[node.lineno - 1]:a]
        lines = ([_from(node, keep)] if keep else []) + [_stmt(dest_mod, here, [_spec(x) for x in taken])]
        if lines: edits.append((a, b, ('\n' + indent).join(lines)))
        else: edits.append((starts[node.lineno - 1], starts[node.end_lineno], ''))
    prefixes = {}
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                if alias.name == src_mod: prefixes[alias.asname or alias.name] = (node, alias)
    for prefix, (node, alias) in prefixes.items():
        root = prefix.split('.')[0]
        reached = [n for n in ast.walk(tree) if isinstance(n, ast.Attribute) and n.attr in moved
                   and ''.join(text[slice(*_at(text, starts, n))].split()) == f'{prefix}.{n.attr}']
        if not reached: continue
        clash = sorted({n.attr for n in reached} & set(bound))
        if clash:
            notes.append(f'{Path(path).name} already binds {", ".join(clash)}; its `{prefix}.` uses were left alone')
            continue
        hit = True
        for n in reached: edits.append((*_at(text, starts, n), n.attr))
        loads = sum(1 for n in ast.walk(tree) if isinstance(n, ast.Name) and n.id == root and isinstance(n.ctx, ast.Load))
        if loads == len(reached):
            if len(node.names) == 1: edits.append((starts[node.lineno - 1], starts[node.end_lineno], ''))
            else: edits.append((*_at(text, starts, node),
                                'import ' + ', '.join(_spec(x) for x in node.names if x is not alias)))
        edits += _add_imports(text, tree, starts, here, {dest_mod: {n.attr for n in reached}}, set())
    if not hit: return None, notes
    return _apply(text, edits), notes

def _gained(path, before, after):
    "Names `after` reads without binding that `before` did not, which is how a move breaks a file."
    return sorted(_free([_parse(path, after)]) - _free([_parse(path, before or '')]))

In [ ]:
#| export
def move_plan(read, src, names, dest, others=(), shim=True):
    "Move `names` from `src` into `dest` and repoint every file in `others` that imported them. `read(path)` gives a file's text, or None."
    src, dest, names, notes = str(src), str(dest), list(names), []
    if not names: raise ValueError('choose a function or class to move')
    if Path(src).resolve() == Path(dest).resolve(): raise ValueError('choose a different file to move into')
    if Path(dest).suffix != '.py': raise ValueError('a move lands in a Python file')
    source = read(src)
    tree, starts = _parse(src, source), _starts(source)
    src_mod, dest_mod = module_of(src), module_of(dest)
    top = {n.name: n for n in tree.body if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef))}
    missing = [n for n in names if n not in top]
    if missing: raise ValueError(f"{', '.join(missing)} is not a top-level function or class in {Path(src).name}")
    moved = [top[n] for n in names]
    spans = sorted(_span(source, starts, n) for n in moved)
    block = ''.join(source[a:b] for a, b in spans).strip('\n') + '\n'
    rest = _apply(source, [(a, b, '') for a, b in spans])
    bound, froms, plains, back = _bound(tree), {}, set(), []
    for name in sorted(_free(moved) - set(names)):
        node = bound.get(name)
        if node is None: continue
        if isinstance(node, (ast.Import, ast.ImportFrom)): _carry(froms, plains, node, name, src_mod)
        else: back.append(name)
    if back: froms.setdefault(src_mod, set()).update(back)

    dtext = read(dest)
    fresh = dtext is None
    dtext = dtext or ''
    dtree, dstarts = _parse(dest, dtext), _starts(dtext)
    clash = [n for n in names if n in _bound(dtree)]
    if clash: raise ValueError(f"{Path(dest).name} already defines {', '.join(clash)}")
    dedits = _add_imports(dtext, dtree, dstarts, dest_mod, froms, plains)
    if (e := _all_edit(dtext, dstarts, dtree, add=names)): dedits.append(e)
    dbody = _apply(dtext, dedits)
    dafter = (dbody.rstrip('\n') + '\n\n\n' + block) if dbody.strip() else block

    rtree, rstarts = _parse(src, rest), _starts(rest)
    used = sorted(n for n in names if n in _free([rtree]))
    want = names if shim else used
    # A re-export the source does not otherwise need is dropped rather than refused: keeping it
    # would import the two modules into each other for nothing.
    if want and back:
        if used: raise ValueError(f'{src_mod} and {dest_mod} would import each other over '
                                  f'{", ".join(back)}; move those as well')
        want = []
        notes.append(f're-export left out: {dest_mod} imports {", ".join(back)} back from {src_mod}')
    sedits = _add_imports(rest, rtree, rstarts, src_mod, {dest_mod: set(want)}, set()) if want else []
    if (e := _all_edit(rest, rstarts, rtree, drop=() if shim else names)): sedits.append(e)
    safter = _apply(rest, sedits)

    rows = [FileEdit(src, source, safter, len(names)),
            FileEdit(dest, None if fresh else dtext, dafter, len(names))]
    for path in others:
        if Path(path).resolve() in (Path(src).resolve(), Path(dest).resolve()): continue
        text = read(path)
        if text is None: continue
        try: after, said = _repoint(path, text, src_mod, dest_mod, names)
        except ValueError as e: notes.append(str(e)); continue
        notes += said
        if after is not None and after != text: rows.append(FileEdit(str(path), text, after, 1))
    for row in rows:
        if (broke := _gained(row.path, row.before, row.after)):
            raise ValueError(f"{Path(row.path).name} would lose {', '.join(broke)}; move what it needs as well")
    return rows, notes

In [ ]:
w = Path(tempfile.mkdtemp())
(w/'shapes.py').write_text('import math\n\n__all__ = [\'area\']\n\ndef area(r): return math.pi * r * r\n')
(w/'geom.py').write_text('"Geometry."\n')
(w/'app.py').write_text('from shapes import area\n\nprint(area(2))\n')
read = lambda p: Path(p).read_text() if Path(p).exists() else None

rows, notes = move_plan(read, w/'shapes.py', ['area'], w/'geom.py', others=[w/'app.py'])
{Path(r.path).name: r.edits for r in rows}

In [ ]:
by = {Path(r.path).name: r for r in rows}
test_eq('def area(r)' in by['geom.py'].after, True)
test_eq('import math' in by['geom.py'].after, True)          # the import the definition needed came too
test_eq('def area(r)' in by['shapes.py'].after, False)
test_eq('from geom import area' in by['shapes.py'].after, True)   # a re-export, by default
test_eq('from geom import area' in by['app.py'].after, True)      # the caller now names the new home

In [ ]:
#: `shim=False` drops the re-export and takes the name out of `__all__` instead.
rows2, _ = move_plan(read, w/'shapes.py', ['area'], w/'geom.py', others=[w/'app.py'], shim=False)
by2 = {Path(r.path).name: r for r in rows2}
test_eq('from geom import area' in by2['shapes.py'].after, False)
test_eq("__all__ = []" in by2['shapes.py'].after, True)

In [ ]:
test_fail(lambda: move_plan(read, w/'shapes.py', [], w/'geom.py'), contains='choose a function or class')
test_fail(lambda: move_plan(read, w/'shapes.py', ['area'], w/'shapes.py'), contains='different file')
test_fail(lambda: move_plan(read, w/'shapes.py', ['area'], w/'geom.txt'), contains='lands in a Python file')
test_fail(lambda: move_plan(read, w/'shapes.py', ['nope'], w/'geom.py'), contains='not a top-level')

In [ ]:
#: A name the destination already binds is refused rather than shadowed.
(w/'taken.py').write_text('def area(r): return 0\n')
test_fail(lambda: move_plan(read, w/'shapes.py', ['area'], w/'taken.py'), contains='already defines')

In [ ]:
#: A definition that still needs a name left behind is refused only when the source still calls it.
(w/'pair.py').write_text('K = 2\n\ndef twice(x): return x * K\n\ndef four(x): return twice(twice(x))\n')
test_fail(lambda: move_plan(read, w/'pair.py', ['twice'], w/'geom.py'), contains='would import each other')

In [ ]:
#: With nothing left calling it, the re-export is dropped instead, and said so.
(w/'p2.py').write_text('K = 2\n\ndef twice(x): return x * K\n')
rows4, notes4 = move_plan(read, w/'p2.py', ['twice'], w/'g2.py')
test_eq(notes4, ['re-export left out: g2 imports K back from p2'])
test_eq('from g2 import twice' in {Path(r.path).name: r for r in rows4}['p2.py'].after, False)

In [ ]:
#: An `import shapes` reached through the module name is repointed too.
(w/'dotted.py').write_text('import shapes\n\nprint(shapes.area(2))\n')
rows3, _ = move_plan(read, w/'shapes.py', ['area'], w/'geom.py', others=[w/'dotted.py'])
by3 = {Path(r.path).name: r for r in rows3}
test_eq('from geom import area' in by3['dotted.py'].after, True)
test_eq('shapes.area' in by3['dotted.py'].after, False)

In [ ]:
#: A file that does not parse is reported by name, not raised through.
(w/'bad.py').write_text('def (:\n')
test_fail(lambda: move_plan(read, w/'bad.py', ['area'], w/'geom.py'), contains='does not parse')